# T1.3 — Whisper WER floor check

Verify that `openai/whisper-small` meets the Phase 3 lower bound:
clean WER < 10%, noisy WER < 25% (pink noise at SNR 10 dB).

If noisy WER > 40%, the ASR -> tactic pipeline is too lossy and we
either drop noisy inputs from the eval or upgrade to whisper-medium.

In [ ]:
# --- VishGuard setup cell (copy from notebooks/00_colabSetup.ipynb) ---
import os, sys, subprocess
from pathlib import Path
REPO_URL = os.environ.get('VISHGUARD_REPO_URL', '')
DRIVE_SRC = '/content/drive/MyDrive/vishguard'
REPO_DIR = Path('/content/vishguard')
ON_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
def sh(cmd, check=True):
    print('$', cmd); subprocess.run(cmd, shell=True, check=check)
if ON_COLAB:
    try:
        from google.colab import drive; drive.mount('/content/drive', force_remount=False)
    except Exception as e: print('Drive mount skipped:', e)
    if REPO_URL:
        if REPO_DIR.exists(): sh(f'cd {REPO_DIR} && git pull --ff-only')
        else: sh(f'git clone {REPO_URL} {REPO_DIR}')
    elif Path(DRIVE_SRC).exists():
        REPO_DIR.mkdir(parents=True, exist_ok=True)
        sh(f'rsync -a --delete --exclude .venv --exclude __pycache__ {DRIVE_SRC}/ {REPO_DIR}/')
    else:
        raise RuntimeError('Set VISHGUARD_REPO_URL or copy the repo to MyDrive/vishguard/')
    sh(f'pip install -q -e {REPO_DIR}')
    sh(f'pip install -q -r {REPO_DIR}/requirements.txt')
    src = str(REPO_DIR / 'src')
    if src not in sys.path: sys.path.insert(0, src)
import vishguard; print('vishguard', vishguard.__version__)

In [ ]:
import numpy as np, torch, librosa
from datasets import load_dataset
from transformers import pipeline
from jiwer import wer

WHISPER_ID = 'openai/whisper-small'
device = 0 if torch.cuda.is_available() else -1
asr = pipeline('automatic-speech-recognition', model=WHISPER_ID, device=device,
               return_timestamps=False)
print('loaded:', WHISPER_ID)

In [ ]:
# 5 clean clips with reference text.
ds = load_dataset('openslr/librispeech_asr', 'clean', split='test', streaming=True,
                  trust_remote_code=True)
clips = []
for i, ex in enumerate(ds):
    if i >= 5: break
    audio = ex['audio']
    samples = audio['array'].astype(np.float32)
    sr = audio['sampling_rate']
    if sr != 16_000:
        samples = librosa.resample(samples, orig_sr=sr, target_sr=16_000)
    clips.append({'samples': samples, 'text': ex['text']})
print(f'clean clips: {len(clips)}  avg_dur={np.mean([len(c["samples"])/16000 for c in clips]):.1f}s')

In [ ]:
def add_pink_noise(samples: np.ndarray, snr_db: float = 10.0) -> np.ndarray:
    # 1/f pink noise via random spectrum shaping.
    n = len(samples)
    spectrum = np.fft.rfft(np.random.randn(n).astype(np.float32))
    freqs = np.fft.rfftfreq(n)
    freqs[0] = freqs[1]
    spectrum /= np.sqrt(freqs)
    noise = np.fft.irfft(spectrum, n=n).astype(np.float32)
    s_pow = float(np.mean(samples ** 2))
    n_pow = float(np.mean(noise ** 2))
    target_n_pow = s_pow / (10 ** (snr_db / 10))
    noise *= np.sqrt(target_n_pow / max(n_pow, 1e-12))
    return samples + noise

noisy_clips = [{'samples': add_pink_noise(c['samples'], 10.0), 'text': c['text']} for c in clips]
print('noisy clips ready')

In [ ]:
def transcribe(clip):
    out = asr({'array': clip['samples'], 'sampling_rate': 16_000})
    return out['text']

def wer_for(subset):
    refs = [c['text'].lower() for c in subset]
    hyps = [transcribe(c).lower() for c in subset]
    return wer(refs, hyps), list(zip(refs, hyps))

clean_wer, clean_pairs = wer_for(clips)
noisy_wer, noisy_pairs = wer_for(noisy_clips)
print(f'clean WER: {clean_wer:.3f}   noisy WER: {noisy_wer:.3f}')
print('acceptance: clean < 0.10, noisy < 0.25 (escalate if noisy > 0.40)')

In [ ]:
# Spot-check: print the pair with the worst alignment per subset.
from jiwer import wer as _wer
def worst(pairs):
    return max(pairs, key=lambda p: _wer(p[0], p[1]))
print('--- worst clean ---')
print('REF:', worst(clean_pairs)[0])
print('HYP:', worst(clean_pairs)[1])
print('--- worst noisy ---')
print('REF:', worst(noisy_pairs)[0])
print('HYP:', worst(noisy_pairs)[1])

## Acceptance checklist

- [ ] clean WER < 0.10
- [ ] noisy WER < 0.25 (escalate to whisper-medium or drop noisy eval if > 0.40)
- [ ] Spot-check worst-pair output looks plausible (no catastrophic failure)